# CIFAR-10 — Bloco 4: Data augmentation

**Motivação.** Os campeões atuais foram treinados sem nenhuma transformação nas imagens. Data augmentation cria
variações plausíveis de cada imagem de treino a cada época, aumentando a diversidade efetiva dos dados sem coletar
novas imagens. É a principal alavanca de regularização que ainda não foi testada.

**Transformações** (aplicadas apenas no treino, direto na GPU; validação e teste usam as imagens originais):
- **RandomCrop 32 com padding 4:** desloca a imagem até 4 pixels em qualquer direção;
- **RandomHorizontalFlip:** espelha horizontalmente com probabilidade 0,5.

**Hipóteses.**
- **CNN:** as convoluções já são invariantes a pequenos deslocamentos, mas o augmentation reduz a memorização do treino.
  Com mais diversidade de dados, (a) o overfitting deve cair, (b) a rede de 4 blocos pode voltar a compensar
  e (c) a necessidade de dropout pode diminuir.
- **MLP:** sem noção de vizinhança entre pixels, cada deslocamento é, para a MLP, uma entrada completamente nova.
  O augmentation pode ensinar alguma invariância, mas o ganho deve ser menor e a convergência bem mais lenta.

**Protocolo.** Receitas fixas iguais aos campeões atuais (CNN: 3 blocos, pooling 2×2, BatchNorm — 79,5% no teste;
MLP: 4×256, Adam 3e-4 — 55,1% no teste), mesmas partições em 5 folds e seed, escolha pela validação e teste revelado
só no fim. Mais épocas e paciência, pois augmentation retarda a convergência. As configurações **sem** augmentation são
re-treinadas no mesmo notebook para uma **comparação pareada por fold** justa.

**Tempo estimado:** ~1h30 com 2× T4 (CNN ~1h15; MLP ~15 min).

**Antes de executar (Kaggle):** *GPU T4 ×2*, *Internet On*, Input **cifar10-python** e secrets `GITHUB_TOKEN` /
`WANDB_API_KEY` anexados. Execute com **Save Version → Save & Run All**.

## 0. Preparação do ambiente

In [ ]:
import os

REPO_URL = "https://github.com/diegoflyra/dfal-neural-networks.git"
REPO_DIR = "/tmp/dfal-neural-networks"  # fora de /kaggle/working: código e dataset não poluem o Output

# Repositório privado: crie o secret GITHUB_TOKEN (Add-ons → Secrets) com um token de leitura do GitHub.
# O token fica só em /tmp (não vai para o Output) e nunca é impresso.
clone_url = REPO_URL
try:
    from kaggle_secrets import UserSecretsClient
    clone_url = REPO_URL.replace("https://", "https://" + UserSecretsClient().get_secret("GITHUB_TOKEN") + "@")
    print("GitHub: usando GITHUB_TOKEN")
except Exception:
    print("GitHub: sem GITHUB_TOKEN (funciona apenas se o repositório for público)")

if os.path.isdir(REPO_DIR):
    !git -C $REPO_DIR pull -q
else:
    !git clone -q $clone_url $REPO_DIR
%cd $REPO_DIR
!git log -1 --oneline

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import shutil
import sys

import pandas as pd

WORKERS_PER_GPU = 2  # experimentos simultâneos por GPU
RESUME_FROM = ""  # ex.: "/kaggle/input/<output-da-versao-anterior>/outputs" para retomar

# CIFAR-10: usa a cópia anexada como Input do Kaggle (segundos), em vez do servidor original (lento).
# A cópia só é aceita se TODOS os arquivos tiverem o MD5 oficial (os mesmos hashes que o torchvision
# usa para validar o download de https://www.cs.toronto.edu/~kriz/cifar.html). Se algo divergir,
# a cópia é descartada e o dataset é baixado do servidor original.
import glob
import hashlib
import tarfile

from torchvision.datasets import CIFAR10

DATA_DIR = os.path.join(REPO_DIR, "data")
CIFAR_DIR = os.path.join(DATA_DIR, "cifar-10-batches-py")
OFFICIAL_MD5 = dict(CIFAR10.train_list + CIFAR10.test_list + [[CIFAR10.meta["filename"], CIFAR10.meta["md5"]]])
os.makedirs(DATA_DIR, exist_ok=True)


def md5(path):
    digest = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def cifar_is_official(folder):
    rows, ok = [], True
    for name, expected in OFFICIAL_MD5.items():
        path = os.path.join(folder, name)
        actual = md5(path) if os.path.isfile(path) else "arquivo ausente"
        ok &= actual == expected
        rows.append({"arquivo": name, "md5 oficial": expected, "md5 da cópia": actual,
                     "status": "OK" if actual == expected else "DIFERENTE"})
    display(pd.DataFrame(rows))
    return ok


if not os.path.isdir(CIFAR_DIR):
    folders = glob.glob("/kaggle/input/**/cifar-10-batches-py", recursive=True)
    archives = glob.glob("/kaggle/input/**/cifar-10-python.tar.gz", recursive=True)
    if folders:
        print(f"Cópia encontrada: {folders[0]}")
        shutil.copytree(folders[0], CIFAR_DIR)
    elif archives:
        print(f"Arquivo encontrado: {archives[0]} | md5 {md5(archives[0])} (oficial: {CIFAR10.tgz_md5})")
        with tarfile.open(archives[0]) as tar:
            tar.extractall(DATA_DIR)
    else:
        print("AVISO: CIFAR-10 não encontrado nos Inputs; será baixado do servidor original (pode levar muitos minutos).")

if os.path.isdir(CIFAR_DIR):
    if cifar_is_official(CIFAR_DIR):
        print("CIFAR-10 verificado: todos os arquivos são idênticos aos oficiais.")
    else:
        shutil.rmtree(CIFAR_DIR)
        print("CÓPIA REJEITADA: arquivos diferentes dos oficiais. O dataset será baixado do servidor original.")

# Onde os resultados são gravados (lido por run_experiment.py, grid_search.py e report_utils.py)
os.environ["EXP_OUTPUT_DIR"] = "/kaggle/working/outputs"
os.makedirs(os.environ["EXP_OUTPUT_DIR"], exist_ok=True)
if RESUME_FROM:
    shutil.copytree(RESUME_FROM, os.environ["EXP_OUTPUT_DIR"], dirs_exist_ok=True)
    print(f"Resultados anteriores copiados de {RESUME_FROM}; o que já foi concluído será pulado.")

# Weights & Biases via Kaggle Secrets; se falhar, os experimentos seguem apenas com o registro local.
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")
    print("W&B: chave carregada.")
except Exception as exc:
    os.environ["WANDB_MODE"] = "disabled"
    print(f"W&B desativado ({exc}). Resultados continuam em {os.environ['EXP_OUTPUT_DIR']}.")

sys.path.insert(0, os.path.join(REPO_DIR, "src"))
import report_utils as rep

In [ ]:
# Verificações rápidas antes de gastar GPU (sem treino):
# GPUs, flags → arquitetura, carregamento em GPU/K-fold (baixa o CIFAR-10) e construção de todas as configurações dos grids
!nvidia-smi -L
!python tests/check_hyperparams.py
!python tests/check_data_loader.py
!python tests/check_grids.py

## Bloco 4 — CNN: augmentation × profundidade × dropout

| Eixo | Valores |
|---|---|
| `augment` | `False`, `True` |
| `conv_blocks` | `3`, `4` |
| `cnn_dropout` | `0.3`, `0.5` |

**8 combinações.** Fixos no bloco: `filters=64`, `filters_growth=double`, `convs_per_block=1`, `kernel_size=3`, `padding=same`, `stride=1`, `pool_size=2`, `fc_neurons=[512]`, `activation=relu`, `cnn_batch_norm=True`, `optimizer=adam`, `lr=0.001`, `momentum=0.9`, `weight_decay=0.0`, `batch_size=128`, `epochs=100`, `patience=10`, `eval_train=True`, `loss_fn=cross_entropy`. Triagem nos folds [1, 2, 3] de 5; as 2 melhores completam os 5 folds. Limite: 20,000,000 parâmetros.

**O que observar:**
- `val/accuracy` com e sem augmentation (heatmaps lado a lado, mesma escala de cores);
- `gap/accuracy`: o augmentation deve reduzir a distância entre treino e validação;
- `melhor_epoca_media`: se ficar perto do limite de 100 épocas, o resultado pode estar limitado pelo orçamento;
- se a rede de 4 blocos e o dropout mais baixo passam a ganhar com augmentation.

In [ ]:
!python src/grid_search.py grids/cnn_b4_augmentation.json --workers_per_gpu 1

In [ ]:
rep.grid_ranking("cnn_b4_augmentation", "triagem")

In [ ]:
rep.heatmap("cnn_b4_augmentation", row="conv_blocks", col="cnn_dropout", facet="augment")

In [ ]:
rep.heatmap("cnn_b4_augmentation", row="conv_blocks", col="cnn_dropout", facet="augment", value="gap/accuracy_mean")

In [ ]:
rep.heatmap("cnn_b4_augmentation", row="conv_blocks", col="cnn_dropout", facet="augment", value="melhor_epoca_media")

In [ ]:
rep.grid_ranking("cnn_b4_augmentation", "final")

In [ ]:
rep.show_champion("cnn_b4_augmentation")
rep.plot_finalists("cnn_b4_augmentation")

#### Comparação pareada por fold (validação)

Referência: a receita campeã **sem** augmentation (3 blocos, dropout 0,5), treinada neste mesmo notebook.
Cada ponto é a diferença em um fold; "(4/5)" indica em quantos folds a configuração venceu a referência.

In [ ]:
rep.paired_comparison("cnn_b4_augmentation__aug0_B3_do0.5", "cnn_b4_augmentation__*")

### 📝 Análise — Bloco 4 — CNN

- **Ganho do augmentation na validação (pareado):** _…_
- **O gap treino-validação caiu?** _…_
- **Com augmentation, 4 blocos passaram a compensar? E o dropout ideal mudou?** _…_
- **Algum resultado ficou limitado pelo orçamento de épocas?** _…_

In [ ]:
# Backup parcial: CSV/JSON/PNG/logs (os .pth ficam em outputs/ na aba Output)
!cd /kaggle/working && zip -qr outputs.zip outputs -x '*.pth' && ls -lh outputs.zip

## Bloco 4 — MLP: augmentation × dropout

| Eixo | Valores |
|---|---|
| `augment` | `False`, `True` |
| `dropout` | `0.0`, `0.2` |

**4 combinações.** Fixos no bloco: `mlp_layers=4`, `mlp_neurons=[256]`, `activation=relu`, `batch_norm=False`, `loss_fn=cross_entropy`, `optimizer=adam`, `lr=0.0003`, `momentum=0.9`, `weight_decay=0.0`, `batch_size=128`, `epochs=150`, `patience=15`, `eval_train=True`. Todas as configurações rodam os 5 folds.

**O que observar:** o ganho (ou perda) com augmentation, o número de épocas necessário e se, com augmentation,
o dropout deixa de ser necessário.

In [ ]:
!python src/grid_search.py grids/mlp_b4_augmentation.json --workers_per_gpu 2

In [ ]:
rep.grid_ranking("mlp_b4_augmentation", "final")

In [ ]:
rep.heatmap("mlp_b4_augmentation", row="dropout", col="augment")

In [ ]:
rep.heatmap("mlp_b4_augmentation", row="dropout", col="augment", value="melhor_epoca_media")

In [ ]:
rep.heatmap("mlp_b4_augmentation", row="dropout", col="augment", value="gap/accuracy_mean")

In [ ]:
rep.show_champion("mlp_b4_augmentation")
rep.plot_finalists("mlp_b4_augmentation")

In [ ]:
rep.paired_comparison("mlp_b4_augmentation__aug0_do0.2", "mlp_b4_augmentation__*")

### 📝 Análise — Bloco 4 — MLP

- **O augmentation ajudou a MLP? Quanto, comparado à CNN?** _…_
- **Quantas épocas a MLP precisou com augmentation?** _…_
- **Com augmentation, o dropout ainda é necessário?** _…_

In [ ]:
# Backup parcial: CSV/JSON/PNG/logs (os .pth ficam em outputs/ na aba Output)
!cd /kaggle/working && zip -qr outputs.zip outputs -x '*.pth' && ls -lh outputs.zip

## Teste revelado

Teste dos campeões do Bloco 4 (escolhidos pela validação), para comparar com os campeões anteriores
(CNN: 79,5% ± 0,4%; MLP: 55,1% ± 0,3%). A comparação pareada no teste é apenas descritiva: a decisão já foi tomada
pela validação.

In [ ]:
rep.final_report(["cnn_b4_augmentation", "mlp_b4_augmentation"])

In [ ]:
cnn_campeao = rep.champion_name("cnn_b4_augmentation")
rep.paired_comparison("cnn_b4_augmentation__aug0_B3_do0.5", [cnn_campeao], metric="test/accuracy")
rep.plot_per_class(["cnn_b4_augmentation__aug0_B3_do0.5", cnn_campeao], metric="recall")

In [ ]:
!cd /kaggle/working && zip -qr outputs.zip outputs -x '*.pth' && ls -lh outputs.zip